# 03 - Training With ALE-Frechet

This notebook shows a minimal training loop where ALE profiles and task similarity are computed during training.

The example is intentionally small: in-memory batches, two epochs, and no filesystem logging.

In [1]:
from pathlib import Path
import sys

repo_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "pyproject.toml").exists() and (path / "src" / "alemtl").exists()
    ),
    Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve(),
)
for path in (repo_root, repo_root / "src"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import torch
from torch import nn

from alemtl.models import MultiTaskModel
from alemtl.similarity import MultiTaskALE, MultitaskSimilarity
from alemtl.training import MultiTaskLoss, MultiTaskTrainer, mae_loss

torch.manual_seed(11)

## Synthetic Task-first Batches

The model sees `(tasks, batch, features)` tensors. Tasks 0 and 1 are similar; task 2 differs.

In [2]:
def make_batches(n_tasks: int = 3, n_batches: int = 4, batch_size: int = 16):
    weights = torch.tensor([[2.0, -1.0], [2.1, -0.9], [-1.0, 2.0]])
    batches = []
    for _ in range(n_batches):
        X = torch.randn(n_tasks, batch_size, 2)
        y = torch.einsum("tbf,tf->tb", X, weights).unsqueeze(-1)
        batches.append((X, y))
    return batches

train_batches = make_batches(n_batches=5)
validation_batches = make_batches(n_batches=3)
test_batches = make_batches(n_batches=3)
train_batches[0][0].shape, train_batches[0][1].shape

(torch.Size([3, 16, 2]), torch.Size([3, 16, 1]))

## Model and Similarity Helpers

`similarity_layers` selects the model slice used by ALE. Here we compare the trunk-plus-head behavior.

In [3]:
model = MultiTaskModel(
    n_tasks=3,
    modules_layout={
        "trunk": {"shared": "hard", "module": lambda: nn.Sequential(nn.Linear(2, 8), nn.ReLU())},
        "head": {"shared": "soft", "module": lambda: nn.Linear(8, 1)},
    },
    similarity_layers={"in": "trunk", "out": "head"},
    same_parameters=True,
)

ale = MultiTaskALE(model, validation_batches, n_tasks=3, n_features_out=1, num_intervals=8, n_guess=32)
similarity = MultitaskSimilarity(ale)
loss = MultiTaskLoss(
    model=model,
    loss_fn=nn.MSELoss(reduction="none"),
    errors_fn={"MAE": mae_loss},
    l2_penalty=1e-4,
)

## Trainer

This schedules ALE and task similarity every epoch. `keep_similarity_epochs=1` keeps the inferred task groups for the next training pass.

In [6]:
trainer = MultiTaskTrainer(
    model=model,
    train_dataloader=train_batches,
    validation_dataloader=validation_batches,
    test_dataloader=test_batches,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-2),
    loss=loss,
    ale=ale,
    multitask_similarity=similarity,
    ale_each_epochs=1,
    similarity_each_epochs=1,
    keep_similarity_epochs=1,
    print_each_epochs=10,
    logging_dir="",
)
trainer.train(epochs=2, max_batches=2)


-------------------------------------------
|                                         |
| EXECUTION INFORMATION                   |
|                                         |
-------------------------------------------
| Learning type:                          |
|                                         |
| Architecture:                           |
| Dataset:                                |
|                                         |
| Number of tasks: 3                      |
| Number of epochs: 2                     |
| Train batch size: 0                     |
| ALE batch size: 0                       |
| Test batch size: 0                      |
| Compute similarity each epochs: 1       |
|                                         |
| Learning rate: 0.0                      |
| L2 penalty: 0.0                         |
|                                         |
| Number of ALE intervals: 0              |
| Early stopping rounds: inf              |
|                              

## Inspect Learned Similarity

After training, the similarity object keeps the latest task-pair scores.

In [7]:
similarity.scores, similarity.tasks_groups()

(tensor([[0.0000, 1.0931, 0.9178],
         [1.0931, 0.0000, 1.0670],
         [0.9178, 1.0670, 0.0000]]),
 (tensor([1.0931, 1.0931, 1.0670]),
  tensor([[0, 1],
          [1, 0],
          [2, 1]])))